# Inteligencia artificial enfocada en la empresa de reparto Manantial

In [1]:
"""
╔══════════════════════════════════════════════════════════╗
║         MANANTIAL IA - Integración Completa RA1          ║
║                                                          ║
║  IL1.1 → Conexión API + LangChain + Streaming + Memoria  ║
║  IL1.2 → Zero-Shot, Few-Shot, Chain-of-Thought           ║
║  IL1.3 → RAG con embeddings y base de datos vectorial    ║
║  IL1.4 → Métricas de evaluación en tiempo real           ║
╚══════════════════════════════════════════════════════════╝


Variables de entorno requeridas:
    GITHUB_TOKEN
"""

import os
import time
import textwrap
from pathlib import Path
from typing import Optional

from dotenv import load_dotenv, find_dotenv

_dotenv_path = find_dotenv(filename=".env", usecwd=True)
if _dotenv_path:
    load_dotenv(_dotenv_path)
else:
    for _dir in (Path.cwd(), *list(Path.cwd().parents)[:8]):
        _f = _dir / ".env"
        if _f.is_file():
            load_dotenv(_f)
            break
    else:
        load_dotenv()

_ls_key = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
if _ls_key:
    os.environ.setdefault("LANGCHAIN_API_KEY", _ls_key)
    os.environ.setdefault("LANGSMITH_API_KEY", _ls_key)

if os.getenv("LANGSMITH_TRACING", "").lower() == "true":
    os.environ["LANGCHAIN_TRACING_V2"] = "true"

import langsmith

if os.getenv("LANGSMITH_TRACING", "").lower() == "true" and _ls_key:
    langsmith.configure(
        enabled=True,
        project_name=os.getenv("LANGSMITH_PROJECT") or "default",
    )

if os.getenv("LANGCHAIN_DEBUG", "").lower() == "true":
    from langchain_core.globals import set_debug

    set_debug(True)

if os.getenv("LANGSMITH_TRACING", "").lower() == "true":
    print(
        "[LangSmith]",
        "proyecto=" + os.getenv("LANGSMITH_PROJECT", "(sin LANGSMITH_PROJECT)"),
        "| api_key=" + ("ok" if _ls_key else "FALTA — sin esto no hay trazas"),
        "| .env=" + (_dotenv_path or "buscado desde cwd"),
    )

# ─────────────────────────────────────────────
# IL1.1 ─ CONEXIÓN API + LANGCHAIN + STREAMING
# ─────────────────────────────────────────────
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

token    = os.environ["GITHUB_TOKEN"]
endpoint = "https://models.github.ai/inference"
model_name = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    base_url=endpoint,
    api_key=token,
    model=model_name,
    temperature=0.3,
    streaming=True          # IL1.1: streaming activado
)

# ─────────────────────────────────────────────
# IL1.2 ─ PROMPT ENGINEERING AVANZADO
#
#  Combina 3 técnicas:
#  1. Zero-Shot  → instrucción directa con rol y restricciones claras
#  2. Few-Shot   → ejemplos concretos de interacciones Manantial
#  3. CoT hint   → "Piensa paso a paso" para consultas complejas
# ─────────────────────────────────────────────

SYSTEM_PROMPT = textwrap.dedent("""\
Eres 'Manantial', el asistente virtual oficial de la empresa de reparto \
Manantial. Tu misión es resolver dudas, gestionar pedidos y entregar \
información precisa sobre servicios de distribución de agua y productos.

## Personalidad
- Amable, claro y resolutivo.
- Usas un lenguaje cercano pero profesional.
- Ante cualquier problema, primero empatizas y luego propones solución.

## Conocimiento base de Manantial
- Servicios: entrega de bidones (10L, 20L), dispensadores en arriendo/venta.
- Frecuencias de reparto: diaria, semanal, quincenal, mensual.
- Zonas de cobertura: revisa siempre el contexto RAG antes de responder.
- Horario de atención: Lunes a Sábado 08:00–18:00.
- Contacto urgente: 600-MANANTIAL.

## Ejemplos de buenas respuestas (Few-Shot)

### Ejemplo 1 — Consulta de precio
Usuario: ¿Cuánto cuesta el bidón de 20 litros?
Asistente: ¡Hola! El bidón de 20 litros tiene un valor de $2.990 con despacho \
incluido en zonas con cobertura activa. Si tienes un dispensador Manantial, \
también puedes optar por el plan mensual con descuento. ¿Te gustaría conocerlo?

### Ejemplo 2 — Reclamo de pedido
Usuario: Mi pedido no llegó y ya son las 15:00.
Asistente: Entiendo tu molestia, eso no debería pasar. Déjame revisar. \
Para gestionar el caso necesito tu número de cliente o RUT. Una vez \
confirmado, reenvío el pedido como prioridad o coordino la entrega para \
el día siguiente sin costo adicional.

### Ejemplo 3 — Pregunta fuera de alcance
Usuario: ¿Cuál es la capital de Francia?
Asistente: Esa pregunta escapa a mi área, soy especialista en los servicios \
de Manantial 💧. Si tienes dudas sobre pedidos, coberturas o productos, \
¡con gusto te ayudo!

## Reglas estrictas
1. Si el usuario pregunta sobre cobertura o stock, SIEMPRE usa primero \
   la información del contexto RAG proporcionado.
2. Si no tienes información suficiente, pide el número de cliente o RUT.
3. Ante quejas, nunca minimices el problema. Empatiza primero.
4. Para consultas complejas (cálculo de costos, comparar planes), \
   razona paso a paso antes de responder (Chain-of-Thought).
5. No inventes precios ni fechas; di que verificarás si no estás seguro.
""")

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# ─────────────────────────────────────────────
# IL1.1 ─ MEMORIA DE CONVERSACIÓN
# ─────────────────────────────────────────────
store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain = prompt | llm

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# ─────────────────────────────────────────────
# IL1.3 ─ RAG: BASE DE CONOCIMIENTO VECTORIAL
# ─────────────────────────────────────────────
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Base de conocimiento interna de Manantial
# (en producción esto vendría de PDFs, BD, etc.)
KNOWLEDGE_BASE = [
    "Manantial cubre las comunas de Santiago Centro, Providencia, Las Condes, "
    "Ñuñoa, Macul, La Florida, Puente Alto y San Bernardo.",

    "El bidón de 10 litros cuesta $1.490 y el de 20 litros $2.990. "
    "Ambos tienen despacho incluido de lunes a sábado.",

    "El dispensador de pie modelo AquaFresh está disponible en arriendo mensual "
    "($4.500/mes) o venta ($89.990). Incluye filtro de purificación.",

    "Plan mensual: 8 bidones de 20L por $19.900 con entrega programada semanal. "
    "Descuento del 10% para clientes con más de 6 meses de antigüedad.",

    "Para cancelar o modificar un pedido, el cliente debe avisar antes de las "
    "19:00 del día anterior llamando al 600-MANANTIAL o por WhatsApp al +56 9 1234 5678.",

    "El tiempo de respuesta ante reclamos es de máximo 24 horas hábiles. "
    "Los reclamos se pueden hacer por teléfono, WhatsApp o en manantial.cl/reclamos.",

    "Manantial no opera en domingos ni festivos. En esos días los pedidos "
    "quedan agendados para el siguiente día hábil.",

    "Los clientes corporativos (empresas con más de 10 pedidos/mes) tienen "
    "ejecutivo dedicado y facturación mensual con 30 días de plazo.",
]

def build_rag_retriever():
    """IL1.3: Carga documentos → chunking → embeddings → vector store."""
    print("🔧 Construyendo base de conocimiento RAG...")

    # IL1.3: Text Chunking (RecursiveCharacterTextSplitter)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=30,
        separators=["\n\n", "\n", ". ", " "]
    )
    docs = [Document(page_content=text) for text in KNOWLEDGE_BASE]
    chunks = splitter.split_documents(docs)
    print(f"   📄 {len(docs)} documentos → {len(chunks)} chunks")

    # IL1.3: Embeddings con modelo local (sin costo de API)
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )

    # IL1.3: Vector Store con FAISS
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    print("   ✅ Vector store listo\n")
    return retriever

# ─────────────────────────────────────────────
# IL1.4 ─ MÉTRICAS DE EVALUACIÓN
# ─────────────────────────────────────────────
class MetricasRAG:
    """
    IL1.4: Evaluación liviana del sistema RAG.
    Métricas medidas:
      - context_retrieved : ¿Se encontró contexto relevante?
      - latency_ms        : Tiempo de respuesta en ms
      - response_length   : Largo de la respuesta
      - session_turns     : Turnos en la conversación (prueba de memoria)
    """
    def __init__(self):
        self.historial: list[dict] = []

    def registrar(self, pregunta: str, contexto: list[str],
                  respuesta: str, latencia_ms: float):
        metricas = {
            "pregunta": pregunta[:60],
            "context_retrieved": len(contexto) > 0,
            "context_docs": len(contexto),
            "latency_ms": round(latencia_ms, 1),
            "response_length": len(respuesta),
        }
        self.historial.append(metricas)
        return metricas

    def resumen(self):
        if not self.historial:
            return
        total = len(self.historial)
        ctx_ok = sum(1 for m in self.historial if m["context_retrieved"])
        avg_lat = sum(m["latency_ms"] for m in self.historial) / total
        print("\n" + "═" * 55)
        print("📊  MÉTRICAS DE EVALUACIÓN (IL1.4)")
        print("═" * 55)
        print(f"  Consultas totales      : {total}")
        print(f"  Contexto RAG recuperado: {ctx_ok}/{total} "
              f"({100*ctx_ok//total}%)")
        print(f"  Latencia promedio      : {avg_lat:.0f} ms")
        print("═" * 55)

metricas = MetricasRAG()

# ─────────────────────────────────────────────
# FUNCIÓN PRINCIPAL: chat con RAG integrado
# ─────────────────────────────────────────────
retriever = None  # se inicializa al primer uso

def chat(mensaje: str, s_id: str = "sesion_1",
         mostrar_contexto: bool = False,
         streaming: bool = True) -> str:
    """
    Pipeline completo RA1:
      1. Recupera contexto RAG (IL1.3)
      2. Inyecta contexto en el prompt (IL1.2)
      3. Genera respuesta con streaming (IL1.1)
      4. Mide métricas (IL1.4)
    """
    global retriever
    if retriever is None:
        retriever = build_rag_retriever()

    t0 = time.perf_counter()

    # IL1.3: Recuperar documentos relevantes
    docs_relevantes = retriever.invoke(mensaje)
    contexto_textos = [d.page_content for d in docs_relevantes]

    if mostrar_contexto:
        print("📂 Contexto RAG recuperado:")
        for i, ctx in enumerate(contexto_textos, 1):
            print(f"   [{i}] {ctx[:100]}...")

    # IL1.2: Enriquecer el mensaje con contexto (RAG augmentation)
    if contexto_textos:
        contexto_str = "\n".join(f"- {c}" for c in contexto_textos)
        mensaje_enriquecido = (
            f"[CONTEXTO INTERNO MANANTIAL]\n{contexto_str}\n\n"
            f"[CONSULTA DEL CLIENTE]\n{mensaje}"
        )
    else:
        mensaje_enriquecido = mensaje

    print(f"\n🔹 Cliente: {mensaje}")
    print("💧 Manantial: ", end="", flush=True)

    # IL1.1: Streaming de respuesta
    respuesta_completa = ""
    if streaming:
        for chunk in conversation.stream(
            {"input": mensaje_enriquecido},
            config={"configurable": {"session_id": s_id}}
        ):
            token_text = chunk.content
            print(token_text, end="", flush=True)
            respuesta_completa += token_text
        print()
    else:
        response = conversation.invoke(
            {"input": mensaje_enriquecido},
            config={"configurable": {"session_id": s_id}}
        )
        respuesta_completa = response.content
        print(respuesta_completa)

    latencia = (time.perf_counter() - t0) * 1000

    # IL1.4: Registrar métricas
    m = metricas.registrar(mensaje, contexto_textos,
                           respuesta_completa, latencia)
    print(f"   ⏱  {m['latency_ms']} ms | "
          f"ctx={m['context_docs']} docs | "
          f"chars={m['response_length']}")
    print("─" * 55)

    return respuesta_completa


# ─────────────────────────────────────────────
# DEMO: prueba de las 4 capas del RA1
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("╔══════════════════════════════════════════════╗")
    print("║                MANANTIAL IA —                ║")
    print("╚══════════════════════════════════════════════╝\n")

    # IL1.1 + Memoria: el asistente recuerda el nombre
    chat("Hola, me llamo Lucas y quiero información sobre sus servicios.")
    chat("¿Recuerdas mi nombre? ¿Qué planes tienen para empresas?")

    # IL1.2 Zero-Shot: pregunta directa
    chat("¿En qué comunas hacen entrega?")

    # IL1.2 Few-Shot / IL1.3 RAG: consulta que usa la base de conocimiento
    chat("¿Cuánto cuesta el bidón de 20 litros y tienen algún plan mensual?",
         mostrar_contexto=True)

    # IL1.2 CoT: problema complejo (el prompt instruye CoT para estos casos)
    chat("Si pido 8 bidones de 20L con el plan mensual y soy cliente hace "
         "8 meses, ¿cuánto pagaría exactamente?")

    # IL1.1 + Memoria: prueba de coherencia conversacional
    chat("¿Puedo cancelar si no me llega el pedido a tiempo?")

    # IL1.4: mostrar resumen de métricas al final
    metricas.resumen()

    # Enviar trazas pendientes a LangSmith antes de cerrar la celda
    try:
        from langchain_core.tracers.langchain import wait_for_all_tracers

        wait_for_all_tracers()
    except Exception:
        pass

[LangSmith] proyecto=proyecto ai | api_key=ok | .env=c:\Users\merii\Desktop\programacion\PruebasConInteligenciaArtificial\.env


c:\Users\merii\Desktop\programacion\PruebasConInteligenciaArtificial\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\merii\Desktop\programacion\PruebasConInteligenciaArtificial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


╔══════════════════════════════════════════════╗
║                MANANTIAL IA —                ║
╚══════════════════════════════════════════════╝

🔧 Construyendo base de conocimiento RAG...
   📄 8 documentos → 8 chunks


C:\Users\merii\AppData\Local\Temp\ipykernel_3620\1650892974.py:210: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7400.12it/s]


   ✅ Vector store listo


🔹 Cliente: Hola, me llamo Lucas y quiero información sobre sus servicios.
💧 Manantial: ¡Hola, Lucas! Encantado de ayudarte. En Manantial ofrecemos servicios de entrega de agua en bidones de 10 litros y 20 litros, con despacho incluido de lunes a sábado. 

Además, contamos con un plan mensual que incluye 8 bidones de 20 litros por solo $19.900, con entrega programada semanal. Si eres cliente con más de 6 meses de antigüedad, puedes acceder a un descuento del 10%.

Si tienes alguna pregunta específica o necesitas más detalles sobre algún servicio en particular, ¡no dudes en decírmelo!
   ⏱  2421.4 ms | ctx=3 docs | chars=502
───────────────────────────────────────────────────────

🔹 Cliente: ¿Recuerdas mi nombre? ¿Qué planes tienen para empresas?
💧 Manantial: ¡Claro, Lucas! Para empresas, ofrecemos un servicio especial que incluye un ejecutivo dedicado y facturación mensual con un plazo de 30 días, ideal para clientes corporativos que realizan más de 10 pedidos 